# Building the Büntgen et al. (2020) chronologies with **dplPy**
### A step-by-step demonstration for dendrochronologists

This notebook rebuilds the nine regional tree-ring-width (TRW) chronologies of

> Büntgen, U. *et al.* (2020) *Prominent role of global volcanism in Common Era
> climate variability and human history.* **Dendrochronologia** 64, 125757.

from nine raw `.rwl` measurement files, using the Python Dendrochronology Program
Library, [**dplPy**](https://github.com/OpenDendro/dplPy).

It is written as a **teaching walk-through**, not production code. If you come from
R's `dplR` or from the ARSTAN program, the steps below should look familiar — the
function names are nearly the same (`readers`, `powt`, `rcs`, `chron`), and each
cell does one small thing you can read and change.

> The *official*, reproducible build is the script
> `build_buntgen_chronologies_dplpy.py` in this repository. This notebook is a
> friendlier companion that shows **how** it works. Both produce the same
> chronologies.

**What we will do**

1. Read one site's ring widths and look at them.
2. Detrend by Regional Curve Standardization (RCS) and build one chronology.
3. See the handful of standardization *choices* Büntgen varied, one at a time.
4. Combine them into the **16-member ensemble** and take its median.
5. Repeat for all nine sites and save the results.

---

## 0. Setup

dplPy installs from PyPI. If you haven't already:

```
pip install dplpy
```

Then the imports — just dplPy plus the usual scientific-Python trio.

In [ ]:
import dplpy as dpl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
import warnings
warnings.simplefilter("ignore")      # keep the notebook tidy; dplPy is chatty about data quirks

ITRDB = Path("itrdb")                 # the nine *_all.rwl files live here
OUT   = Path("output"); OUT.mkdir(exist_ok=True)

SITES = ["GTB","SCO","QUL","SCA","ALP","YAM","TAI","ALT","NYA"]
print("dplPy version:", dpl.__version__)

## 1. Read a site and look at it

We'll learn the method on one site — **TAI** (Taimyr, northern Siberia) — because
it is small and fast. Reading a Tucson `.rwl` file is one call, `dpl.readers()`,
and it returns a familiar table: **years down the rows, one column per measured
series**.

In [ ]:
rwl = dpl.readers(str(ITRDB / "TAI_all.rwl"))
print(f"{rwl.shape[1]} series, {int(rwl.index.min())} to {int(rwl.index.max())} CE")
rwl.iloc[:5, :4]        # a corner of the table: years x series

Each column is one radius of one tree, in units of 0.01 mm. Plotting a few shows
the classic shape: wide rings when the tree is young, narrowing with age. That
**biological age trend** is what standardization must remove so that only the
common climate signal remains.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.2))
for col in rwl.columns[:25]:
    ax.plot(rwl.index, rwl[col], lw=0.5, alpha=0.5)
ax.set(xlabel="Year (CE)", ylabel="Ring width (0.01 mm)",
       title="TAI — 25 raw ring-width series")
plt.show()

## 2. One chronology, by Regional Curve Standardization (RCS)

**RCS** aligns every series by *cambial age* (rings counted from the pith), averages
them into a single **regional curve** describing the typical age trend, and divides
each series by that curve. The result is a set of dimensionless **ring-width indices**
that hover around 1, with the age trend removed but the long-timescale climate signal
retained.

RCS needs to know each series' **pith offset** — how many rings are missing between
the pith and the first measured ring. These files don't carry that information, so we
set it to 1 for every series (first measured ring = cambial age 1). *(This is a known
limitation for old relict wood; see the note at the end.)*

In [ ]:
# pith offset = 1 for every series
po = pd.DataFrame({"series": rwl.columns,
                   "pith_offset": np.ones(rwl.shape[1], dtype=int)})

# detrend by RCS -> ring-width indices (one call)
rwi = dpl.rcs(rwl, po=po, make_plot=False)

# average the indices into a chronology (robust biweight mean, like dplR/ARSTAN)
crn = dpl.chron(rwi, biweight=True, plot=False)
crn.head()

`dpl.chron()` returns the chronology (`std`) indexed by year, plus the **sample
depth** (`samp_depth`) — how many series contribute each year. Let's plot it. Notice
how noisy the early part is, where only a few series exist.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.axhline(1.0, color="0.7", lw=0.8)
ax.plot(crn.index, crn["std"], color="firebrick", lw=0.6)
ax.set(xlabel="Year (CE)", ylabel="RWI", title="TAI — standard RCS chronology")
ax2 = ax.twinx()
ax2.fill_between(crn.index, crn["samp_depth"], color="steelblue", alpha=0.15)
ax2.set_ylabel("sample depth"); plt.show()

That is already a usable chronology. Büntgen didn't stop at one, though — they
built an **ensemble** of 16 slightly different ones and took the median, so that no
single methodological choice dominates. The next cells show those choices one at a
time.

## 3. The standardization choices, one at a time

### 3a. Ratios vs. power transformation

Dividing by the regional curve gives **ratio** indices (above). An alternative is to
first apply the **Cook & Peters (1997) power transformation** — which stabilizes the
variance — and then take **residuals** (subtract the curve) instead of ratios.

One subtlety worth understanding: residual indices come out centered on **0**, while
ratio indices center on **1**. ARSTAN (which Büntgen used) quietly rescales the
residuals back onto the ratio scale so every ensemble member is comparable. We do the
same, explicitly, so the later median is meaningful.

In [ ]:
# power-transform, then RCS with residuals (ratios=False)
rwl_pt = dpl.powt(rwl, method="cook")
rwi_pt = dpl.rcs(rwl_pt, po=po, ratios=False, make_plot=False)
crn_pt = dpl.chron(rwi_pt, biweight=True, plot=False)

print("ratio chronology mean   :", round(crn["std"].mean(), 3))      # ~1
print("residual chronology mean:", round(crn_pt["std"].mean(), 3))   # ~0  <-- different scale!

Rescaling the residual chronology onto the mean and standard deviation of its
ratio counterpart puts both on the same ~1 scale (this mimics ARSTAN's `calind`):

In [ ]:
def rescale_to(residual, reference):
    """Put a residual (~0) chronology onto the mean & SD of a reference (~1) one,
    over their common years."""
    yrs = residual.index.intersection(reference.index)
    r, f = residual.reindex(yrs), reference.reindex(yrs)
    ok = r.notna() & f.notna()
    return (residual - r[ok].mean()) / r[ok].std() * f[ok].std() + f[ok].mean()

crn_pt_scaled = rescale_to(crn_pt["std"], crn["std"])
print("rescaled residual chronology mean:", round(crn_pt_scaled.mean(), 3))   # now ~1

### 3b. Signal-free RCS

Ordinary RCS can distort low-frequency signal when the sample's age structure changes
over time. **Signal-free RCS** (Melvin & Briffa 2014) corrects this by iterating. In
dplPy it is one call, `dpl.sfrcs()` — a Python port of the CRUST algorithm, so you no
longer need separate software for it.

In [ ]:
rwi_sf  = dpl.sfrcs(rwl, po=po, ratios=True, make_plot=False, verbose=False)
crn_sf  = dpl.chron(rwi_sf, biweight=True, plot=False)
print("signal-free RCS chronology mean:", round(crn_sf["std"].mean(), 3))

### 3c. Standard vs. ARSTAN chronology

Finally, the indices can be averaged into a plain **standard** chronology
(`dpl.chron`) or an **ARSTAN** chronology (`dpl.chron_ars`), which removes residual
autocorrelation and then re-introduces the common persistence. We use the `ars`
column from `chron_ars`.

In [ ]:
crn_ars = dpl.chron_ars(rwi, biweight=True, verbose=False)   # std / res / ars / samp_depth
print("columns:", list(crn_ars.columns))

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.axhline(1.0, color="0.7", lw=0.8)
ax.plot(crn.index, crn["std"], color="firebrick", lw=0.6, label="standard (STD)")
ax.plot(crn_ars.index, crn_ars["ars"], color="darkorange", lw=0.6, alpha=0.8, label="ARSTAN (ARS)")
ax.set(xlabel="Year (CE)", ylabel="RWI", title="TAI — STD vs ARSTAN"); ax.legend(); plt.show()

## 4. The 16-member ensemble

Crossing those choices gives Büntgen's ensemble. One more detail: the regional-curve
smoothing comes in two flavours — the **default** curve, and a **"−10"** curve
(a cubic spline at 10 % of the curve's length). Power-transform members always use the
−10 curve. Put together, the 16 members (run on **all** series and again on a
**living/relict** split, below) are:

| | variance stab. | index | RC smoothing | chronology |
|-|-|-|-|-|
| ptRCS-10 ×2 | power transform | residual→rescaled | −10 | STD, ARS |
| RCS ×2 | none | ratio | default | STD, ARS |
| RCS-10 ×2 | none | ratio | −10 | STD, ARS |
| sfptRCS10 | power transform | residual→rescaled | −10 | signal-free |
| sfRCS10 | none | ratio | −10 | signal-free |

Rather than write those out 16 times, here is **one small, readable function** that
builds any single member from the choices, using exactly the calls we met above.

In [ ]:
def build_member(rwl, power_transform, smoothing, chron_kind, signal_free):
    """Build one ensemble member and return its chronology (a year-indexed Series).

    power_transform : True -> powt + residuals ; False -> ratios
    smoothing       : "default" or "ten"  (ten = 10%-of-length spline)
    chron_kind      : "STD" or "ARS"
    signal_free     : True -> sfrcs() ; False -> rcs()
    """
    po = pd.DataFrame({"series": rwl.columns,
                       "pith_offset": np.ones(rwl.shape[1], dtype=int)})

    # (1) optional power transform; sets whether we take residuals or ratios
    if power_transform:
        data, ratios = dpl.powt(rwl, method="cook"), False
    else:
        data, ratios = rwl, True

    # (2) detrend by RCS (ordinary or signal-free)
    if signal_free:
        rwi = dpl.sfrcs(data, po=po, ratios=ratios, make_plot=False, verbose=False)
    elif smoothing == "ten":
        # nyrs=None is dplPy's 10%-of-length spline -> the "-10" curve
        rwi = dpl.rcs(data, po=po, ratios=ratios, nyrs=None, method="caps", make_plot=False)
    else:
        rwi = dpl.rcs(data, po=po, ratios=ratios, method="ads", make_plot=False)   # default curve

    # (3) average into a chronology
    if chron_kind == "ARS":
        crn = dpl.chron_ars(rwi, biweight=True, verbose=False)["ars"]
    else:
        crn = dpl.chron(rwi, biweight=True, plot=False)["std"]
    return crn.replace([np.inf, -np.inf], np.nan)

Now list the eight member *recipes* (we'll add the living/relict versions in the
full run) and build them for TAI. The power-transform/signal-free members get rescaled
onto the ratio scale, as in §3a.

In [ ]:
# name : (power_transform, smoothing, chron_kind, signal_free)
RECIPES = {
    "ptRCS-10_STD": (True,  "ten",     "STD", False),
    "ptRCS-10_ARS": (True,  "ten",     "ARS", False),
    "RCS_STD":      (False, "default", "STD", False),
    "RCS_ARS":      (False, "default", "ARS", False),
    "RCS-10_STD":   (False, "ten",     "STD", False),
    "RCS-10_ARS":   (False, "ten",     "ARS", False),
    "sfptRCS10":    (True,  "ten",     "STD", True),
    "sfRCS10":      (False, "ten",     "STD", True),
}
# members whose residual scale must be matched to a ratio member:
RESCALE_TO = {"ptRCS-10_STD": "RCS-10_STD", "ptRCS-10_ARS": "RCS-10_ARS",
              "sfptRCS10": "sfRCS10"}

members = {name: build_member(rwl, *opts) for name, opts in RECIPES.items()}
for name, ref in RESCALE_TO.items():
    members[name] = rescale_to(members[name], members[ref])

print("built", len(members), "members for TAI")
pd.Series({k: round(v.mean(), 3) for k, v in members.items()}, name="member mean")

All eight now sit near 1. Plot them together with their **median** — the single
chronology Büntgen carries forward. The members agree closely (they are "16 slightly
different realisations"), which is the whole point of the ensemble: the median is
robust to any one choice.

In [ ]:
ens = pd.DataFrame(members)
med = ens.median(axis=1)

fig, ax = plt.subplots(figsize=(10, 3.4))
ax.axhline(1.0, color="0.7", lw=0.8)
for col in ens.columns:
    ax.plot(ens.index, ens[col], lw=0.4, alpha=0.4)
ax.plot(med.index, med, color="black", lw=0.9, label="ensemble median")
ax.set(xlabel="Year (CE)", ylabel="RWI", title="TAI — ensemble members (grey) and median (black)")
ax.legend(); plt.show()

## 5. Living vs. relict, and the full ensemble

Büntgen also ran each recipe **twice**: once on all series, and once splitting the
collection into **living** trees (last ring after 1950) and **relict** wood, building
a chronology for each and merging them with equal weight. That doubles 8 → **16
members**. The helper below does the split-and-merge; it's the only extra piece, and
it just calls `build_member` on each subset.

In [ ]:
def build_all_16(rwl):
    """Build all 16 members for one site: the 8 recipes on all series, and again
    on a living/relict split, merged. Returns a DataFrame (years x members)."""
    def members_for(subset):
        m = {n: build_member(subset, *o) for n, o in RECIPES.items()}
        for name, ref in RESCALE_TO.items():
            m[name] = rescale_to(m[name], m[ref])
        return m

    out = {f"all_{n}": c for n, c in members_for(rwl).items()}

    last = rwl.apply(lambda s: s.last_valid_index())
    living, relict = rwl.loc[:, last > 1950], rwl.loc[:, last <= 1950]
    if living.shape[1] >= 5 and relict.shape[1] >= 5:
        liv, rel = members_for(living), members_for(relict)
        for n in RECIPES:                          # simple equal-weight average of the two
            out[f"lr_{n}"] = pd.concat([liv[n], rel[n]], axis=1).mean(axis=1)
    else:                                          # too few to split -> reuse the all-series members
        for n in RECIPES:
            out[f"lr_{n}"] = out[f"all_{n}"]
    return pd.DataFrame(out)

## 6. All nine sites

Now the pay-off: nine `.rwl` files in, nine ready-to-use chronologies out. The loop
below reads each site, builds its 16 members, takes the median, and saves it. The
small sites take seconds; the largest (**QUL**, **NYA**) take a few minutes because
the signal-free members iterate — so the whole loop runs in roughly ten minutes.

In [ ]:
chronologies = {}
for code in SITES:
    rwl_site = dpl.readers(str(ITRDB / f"{code}_all.rwl"))
    ens16 = build_all_16(rwl_site)
    median = ens16.median(axis=1)
    chronologies[code] = median
    ens16.to_csv(OUT / f"{code}_all_variants.csv", index_label="year")   # all 16 members
    print(f"{code}: {rwl_site.shape[1]:5d} series -> median over "
          f"{int(median.index.min())}..{int(median.index.max())} CE")

# one tidy table of the nine medians, saved for downstream use
all_crns = pd.DataFrame(chronologies).sort_index()
all_crns.to_csv(OUT / "buntgen_all_chronologies.csv", index_label="year")
print("\nsaved", len(chronologies), "chronologies to", OUT)

### The nine chronologies

In [ ]:
fig, axes = plt.subplots(len(SITES), 1, figsize=(11, 13), sharex=True)
for ax, code in zip(axes, SITES):
    c = chronologies[code].dropna()
    ax.axhline(1.0, color="0.7", lw=0.6)
    ax.plot(c.index, c.values, color="firebrick", lw=0.4)
    ax.set_ylabel(code, rotation=0, ha="right", va="center")
    ax.set_ylim(0, 2)
axes[-1].set_xlabel("Year (CE)")
axes[0].set_title("Büntgen et al. (2020) regional TRW chronologies, rebuilt with dplPy")
plt.tight_layout(); plt.show()

## 7. What you have, and one caveat

Each `output/<CODE>_all_variants.csv` holds the 16 member chronologies for a site, and
`output/buntgen_all_chronologies.csv` holds the nine medians side by side — ready for
climate calibration, reconstruction, or comparison.

**The pith-offset caveat.** We assumed a pith offset of 1 for every series because the
archived files don't record it. For old **relict** wood whose pith is long gone, the
true cambial age at the first ring can be large, and assuming 1 biases the regional
curve. This shows up most at **NYA**, whose thin, relict-dominated early record sits a
little high — a feature visible in Büntgen's own published reconstruction too, so it is
inherent to the data rather than to dplPy. Supplying real pith offsets (via the
`pith_offset` column) would improve that site.

**That's the whole method** — `readers`, `powt`, `rcs`/`sfrcs`, `chron`/`chron_ars`,
a median — nine raw files to nine chronologies, entirely in open-source Python. For the
exact, reproducible build (identical results, more error-checking), see
`build_buntgen_chronologies_dplpy.py` in this repository.